In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split,GridSearchCV
#GridSearchCV- Searches the best combination of hyperparameters using cross-validation
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [99]:
df=pd.read_csv('Churn_Modelling.csv')

In [100]:
df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)
label_encoder1=LabelEncoder()
df['Gender']=label_encoder1.fit_transform(df['Gender'])

one_hot_encoder1=OneHotEncoder(handle_unknown='ignore')
geo_encoded=one_hot_encoder1.fit_transform(df[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encoder1.get_feature_names_out(['Geography']))

df=pd.concat([df.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [101]:
X=df.drop('Exited',axis=1)
y=df['Exited']

In [102]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler1=StandardScaler()
X_train=scaler1.fit_transform(X_train)
X_test=scaler1.transform(X_test)

In [103]:
with open('label_encoder1.pkl','wb') as file:
    pickle.dump(label_encoder1,file)
with open('one_hot_encoder1.pkl','wb') as file:
    pickle.dump(one_hot_encoder1,file)
with open('scaler1.pkl','wb') as file:
    pickle.dump(scaler1,file)

In [104]:
#define a function to create a model and try different parameters (KerasClassifier)
def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [ ]:
#create a Keras Classifier
model=KerasClassifier(model=create_model,verbose=0)
#wraps the keras model so it can we interact with Scikit-learn tools like GridSearchCV


In [ ]:
#define the grid search parameters
param_grid={
        'model__neurons':[8,16,32],
        'model__layers':[1,2],
        'epochs':[20,50],  #how many times the model goes thr the entire dataset
        'batch_size':[10,20]  #number of samples per training update
}

In [107]:
#perform grid search
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,verbose=1)
grid_result=grid.fit(X_train,y_train)

#display result
print("Best Accuracy: %.4f using %s"% (grid_result.best_score_,grid_result.best_params_))

Fitting 3 folds for each of 24 candidates, totalling 72 fits


c:\Users\USER\Documents\project\ANN Classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.8584 using {'batch_size': 10, 'epochs': 50, 'model__layers': 1, 'model__neurons': 8}
